1️⃣ Match Data Collection


In [1]:
import requests
import pandas as pd
import json
import time
from datetime import timedelta

In [2]:
ESPN_SCOREBOARD_URL = (
    "https://site.api.espn.com/apis/site/v2/sports/soccer/ksa.1/scoreboard"
)

OPEN_METEO_URL = (
    "https://archive-api.open-meteo.com/v1/archive"
)

START_DATE = pd.Timestamp("2022-08-25")
END_DATE = pd.Timestamp("2026-09-13")

TIMEZONE = "Asia/Riyadh"

2️⃣ Data Cleaning


In [3]:
SEASON_TEAMS = {

    "2022-23": [
        "Abha",
        "Al Adalah",
        "Al Batin",
        "Al Ettifaq",
        "Al Fateh",
        "Al Fayha",
        "Al Hilal",
        "Al Ittihad",
        "Al Khaleej",
        "Al Nassr",
        "Al Raed",
        "Al Shabab",
        "Al Taawoun",
        "Al Tai",
        "Al Wehda",
        "Damac"
    ],

    "2023-24": [
        "Abha",
        "Al Ahli",
        "Al Akhdoud",
        "Al Ettifaq",
        "Al Fateh",
        "Al Fayha",
        "Al Hazem",
        "Al Hilal",
        "Al Ittihad",
        "Al Khaleej",
        "Al Nassr",
        "Al Raed",
        "Al Riyadh",
        "Al Shabab",
        "Al Taawoun",
        "Al Tai",
        "Al Wehda",
        "Damac"
    ],

    "2024-25": [
        "Al Ahli",
        "Al Ettifaq",
        "Al Fateh",
        "Al Fayha",
        "Al Hilal",
        "Al Ittihad",
        "Al Kholood",
        "Al Nassr",
        "Al Qadsiah",
        "Al Raed",
        "Al Riyadh",
        "Al Shabab",
        "Al Taawoun",
        "Al Okhdood",
        "Al Orubah",
        "Damac",
        "Al Khaleej",
        "Al Wehda"
    ],

    "2025-26": [
        "Al Ahli",
        "Al Ettifaq",
        "Al Fateh",
        "Al Fayha",
        "Al Hilal",
        "Al Ittihad",
        "Al Kholood",
        "Al Nassr",
        "Al Qadsiah",
        "Al Riyadh",
        "Al Shabab",
        "Al Taawoun",
        "Al Okhdood",
        "Damac",
        "Al Khaleej",
        "Al Hazem",
        "Al Najma",
        "NEOM"
    ],

    "2026-27": [
        "Abha",
        "Al Ahli",
        "Al Diriyah",
        "Al Ettifaq",
        "Al Faisaly",
        "Al Fateh",
        "Al Fayha",
        "Al Hazem",
        "Al Hilal",
        "Al Ittihad",
        "Al Khaleej",
        "Al Kholood",
        "Al Nassr",
        "Al Qadsiah",
        "Al Riyadh",
        "Al Shabab",
        "Al Taawoun",
        "NEOM"
    ]
}

In [4]:
CLUB_CITY = {

    "Abha": "Abha",
    "Al Adalah": "Al Ahsa",
    "Al Ahli": "Jeddah",
    "Al Batin": "Hafr Al Batin",
    "Al Ettifaq": "Dammam",
    "Al Faisaly": "Harmah",
    "Al Fateh": "Al Ahsa",
    "Al Fayha": "Al Majma'ah",
    "Al Hazem": "Ar Rass",
    "Al Hilal": "Riyadh",
    "Al Ittihad": "Jeddah",
    "Al Khaleej": "Saihat",
    "Al Kholood": "Ar Rass",
    "Al Nassr": "Riyadh",
    "Al Najma": "Unaizah",
    "Al Qadsiah": "Khobar",
    "Al Raed": "Buraidah",
    "Al Riyadh": "Riyadh",
    "Al Shabab": "Riyadh",
    "Al Taawoun": "Buraidah",
    "Al Tai": "Ha'il",
    "Al Wehda": "Makkah",
    "Damac": "Khamis Mushait",
    "Al Akhdoud": "Najran",
    "Al Okhdood": "Najran",
    "Al Orubah": "Sakakah",
    "NEOM": "Tabuk",
    "Al Diriyah": "Diriyah"
}

In [6]:
CITY_COORDS = {

    "Riyadh": (24.7136, 46.6753),
    "Jeddah": (21.4858, 39.1925),
    "Dammam": (26.4207, 50.0888),
    "Al Ahsa": (25.3830, 49.5880),
    "Buraidah": (26.3592, 43.9818),
    "Makkah": (21.3891, 39.8579),
    "Abha": (18.2164, 42.5053),
    "Khamis Mushait": (18.3000, 42.7333),
    "Hafr Al Batin": (28.4328, 45.9708),
    "Ar Rass": (25.8694, 43.4973),
    "Al Majma'ah": (25.9100, 45.3560),
    "Khobar": (26.2172, 50.1971),
    "Saihat": (26.4836, 50.0400),
    "Harmah": (25.2631, 45.3397),
    "Ha'il": (27.5114, 41.7208),
    "Najran": (17.4933, 44.1277),
    "Sakakah": (29.9697, 40.2064),
    "Unaizah": (26.0843, 43.9930),
    "Tabuk": (28.3838, 36.5550),
    "Diriyah": (24.7136, 46.6753)
}

In [7]:
def normalize_team_name(name):

    replacements = {

        "Al-Ahli": "Al Ahli",
        "Al-Ittihad": "Al Ittihad",
        "Al-Nassr": "Al Nassr",
        "Al-Hilal": "Al Hilal",
        "Al-Shabab": "Al Shabab",
        "Al-Taawoun": "Al Taawoun",
        "Al-Raed": "Al Raed",
        "Al-Fateh": "Al Fateh",
        "Al-Faisaly": "Al Faisaly",
        "Al-Fayha": "Al Fayha",
        "Al-Wehda": "Al Wehda",
        "Al-Ettifaq": "Al Ettifaq",
        "Al-Batin": "Al Batin",
        "Al-Hazem": "Al Hazem",
        "Al-Qadsiah": "Al Qadsiah",
        "Al-Khaleej": "Al Khaleej",
        "Al-Kholood": "Al Kholood",
        "Al-Tai": "Al Tai",
        "Al-Akhdoud": "Al Akhdoud",
        "Al-Okhdood": "Al Okhdood",
        "Al-Orubah": "Al Orubah",
        "Al-Adalah": "Al Adalah",
        "Al-Riyadh": "Al Riyadh",
        "Al-Najma": "Al Najma",
        "NEOM SC": "NEOM",
        "Neom": "NEOM"
    }

    return replacements.get(name, name)

In [8]:
def get_season(date):

    if date.month >= 8:
        return f"{date.year}-{str(date.year + 1)[-2:]}"

    return f"{date.year - 1}-{str(date.year)[-2:]}"

In [9]:
def get_matches_from_espn(start_date, end_date):

    all_matches = []

    current_date = start_date

    while current_date <= end_date:

        date_str = current_date.strftime("%Y%m%d")

        params = {
            "dates": date_str
        }

        try:

            response = requests.get(
                ESPN_SCOREBOARD_URL,
                params=params,
                timeout=30
            )

            if response.status_code != 200:

                print(
                    f"ERROR {date_str}: "
                    f"{response.status_code}"
                )

                current_date += timedelta(days=1)
                continue

            data = response.json()

            events = data.get("events", [])

            for event in events:

                competitions = event.get(
                    "competitions",
                    []
                )

                if not competitions:
                    continue

                competition = competitions[0]

                competitors = competition.get(
                    "competitors",
                    []
                )

                if len(competitors) < 2:
                    continue

                home = None
                away = None
                home_score = None
                away_score = None

                for team in competitors:

                    team_name = team.get(
                        "team", {}
                    ).get(
                        "displayName"
                    )

                    team_name = normalize_team_name(
                        team_name
                    )

                    if team.get("homeAway") == "home":

                        home = team_name
                        home_score = team.get("score")

                    elif team.get("homeAway") == "away":

                        away = team_name
                        away_score = team.get("score")


                if not home or not away:
                    continue


                venue = competition.get(
                    "venue",
                    {}
                )

                address = venue.get(
                    "address",
                    {}
                )


                all_matches.append({

                    "event_id": event.get("id"),

                    "season": get_season(
                        current_date
                    ),

                    "date": current_date.strftime(
                        "%Y-%m-%d"
                    ),

                    "datetime_utc": event.get(
                        "date"
                    ),

                    "home_team": home,

                    "away_team": away,

                    "home_score": home_score,

                    "away_score": away_score,

                    "venue": venue.get(
                        "fullName"
                    ),

                    "venue_city": address.get(
                        "city"
                    ),

                    "venue_country": address.get(
                        "country"
                    )
                })


            print(
                f"{date_str} -> "
                f"{len(events)} events"
            )


        except Exception as e:

            print(
                f"ERROR {date_str}: {e}"
            )


        current_date += timedelta(days=1)

        time.sleep(0.05)


    return pd.DataFrame(all_matches)

In [11]:
matches_df = get_matches_from_espn(
    START_DATE,
    END_DATE
)

print("Total matches:", len(matches_df))

display(matches_df.head(10))

20220825 -> 2 events
20220826 -> 3 events
20220827 -> 3 events
20220828 -> 0 events
20220829 -> 0 events
20220830 -> 0 events
20220831 -> 2 events
20220901 -> 3 events
20220902 -> 2 events
20220903 -> 2 events
20220904 -> 1 events
20220905 -> 0 events
20220906 -> 0 events
20220907 -> 0 events
20220908 -> 3 events
20220909 -> 2 events
20220910 -> 2 events
20220911 -> 0 events
20220912 -> 0 events
20220913 -> 0 events


KeyboardInterrupt: 